In [16]:
from imutils.video import FPS
import numpy as np
import time
import os
import urllib
import requests, json
import pickle
from collections import deque, Counter
import torch
from my_utils.base import Resnet50
from my_utils.transform import make_transform
import cv2 as cv
from PIL import Image
from collections import deque, Counter
import socketio
import queue
import ast


In [17]:
# SocketIO setup
sio = socketio.Client()
q = queue.Queue()


In [18]:
# Load custom face recognition model on CPU
model = Resnet50(embedding_size=512)
checkpoint = torch.load("/home/devp/Downloads/FR.pth", map_location=torch.device("cpu"))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.to(torch.device("cpu"))
print("Model loaded")


Model loaded


/tmp/ipykernel_16911/686755082.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("/home/devp/Downloads/FR.pth", map_location=torch.device("cpu"))


In [19]:
base_url = "http://127.0.0.1:8000/"
file_path = os.path.dirname(os.path.abspath('')) + os.sep

def get_all_knn():
    URL = base_url + "dashboard/api/get_face_codings_testknn"

    with urllib.request.urlopen(URL) as url:
        data = json.loads(url.read().decode())
        

    all_sig = deque()
    id_name = {}
    emp_data = []
    person_id = []

    for d in data:
        sig = d["face_feature"]
        p_id = d["person_id"]
        emp_data.append(d["person_name"] + "("+ d["department_name"] +")")
        id_name[p_id] = d["person_name"] + "("+ d["department_name"] +")"
        person_id.append(p_id)
        all_sig.append(sig)

    return emp_data, all_sig, person_id, id_name

# Load KNN data
person_names, known_face_encodings, ids_for_person_ids, knn_id_n = get_all_knn()

In [20]:
len(person_names)

12819

In [21]:
len(known_face_encodings)

12819

In [23]:
def minkowski_distance(a, b, p):
    # Convert both arrays to float for numeric operations
    a = np.array(a, dtype=np.float32)
    
    b = np.array(b, dtype=np.float32)
   
    
    # Calculate Minkowski distance
    distance=np.power(np.sum(np.abs(a - b)**p), 1/p)
    
    return distance

In [27]:
 # KNN classifier implementation
def knn_classifier(unknown_encodings, known_face_encodings, ids_for_person_ids, k=3, p=2, threshold=0.5):
    distances = np.array([minkowski_distance(unknown_encodings, ast.literal_eval(encoding), p) for encoding in known_face_encodings])
    print(distances)
    filtered_indices = np.where(distances < threshold)
    filtered_distances = distances
    [filtered_indices]
    

    

    
    if filtered_distances.any():
        try:
            closest_indices = np.argpartition(filtered_distances, k)[:k]
            closest_person_ids = [ids_for_person_ids[i] for i in filtered_indices[0][closest_indices]]
            person_id_counts = Counter(closest_person_ids)
            most_common_person_id, _ = person_id_counts.most_common(1)[0]
            print(most_common_person_id, "try condition")
           
            return most_common_person_id, filtered_distances[closest_indices]
        except:
            closest_indices = np.argpartition(distances, k)[:k]
            closest_person_ids = [ids_for_person_ids[i] for i in closest_indices]
            person_id_counts = Counter(closest_person_ids)
            most_common_person_id, _ = person_id_counts.most_common(1)[0]
            print(most_common_person_id, "except6 condition")
            
            return most_common_person_id, distances[closest_indices]
    else:
        return 'unknown', 0


In [28]:
# SSD model loading
prototxt_file = 'Resnet_SSD_deploy.prototxt'

caffemodel_file = 'Res10_300x300_SSD_iter_140000.caffemodel'
net = cv.dnn.readNetFromCaffe(prototxt_file, caffeModel=caffemodel_file)
print('ResNetSSD model loaded successfully')
thresholdfordetection = 0.5

ResNetSSD model loaded successfully


In [40]:

frame = cv.imread("/home/devp/dataset/1220/11260.jpg") 
   
    
small_frame = cv.resize(frame, (0, 0), fx=0.65, fy=0.65, interpolation=cv.INTER_AREA)
origin_h, origin_w = small_frame.shape[:2]
blob = cv.dnn.blobFromImage(small_frame)
net.setInput(blob)
detections = net.forward()

face_locations = []
face_encodings = []
face_names = []

for i in range(0, detections.shape[2]):
    confidence = detections[0, 0, i, 2]
    if confidence > thresholdfordetection:
        bounding_box = detections[0, 0, i, 3:7] * np.array([origin_w, origin_h, origin_w, origin_h])
        left, top, right, bottom = bounding_box.astype('int')
        face_frame = np.ascontiguousarray(frame[max(0, top-60):min(origin_h, bottom+60), max(0, left-60):min(origin_w, right+60)])
        
        pil_img = Image.fromarray(face_frame)
        transformed_img = make_transform(is_train=False)(pil_img).unsqueeze(0)  # Apply transformation

        # Extract embeddings from the model
        embedding = model(transformed_img).detach().cpu().numpy()[0]

        face_locations.append((left, top, right, bottom))
        face_encodings.append(embedding)

        # Use KNN to classify the face
        person_id, _ = knn_classifier(embedding, known_face_encodings, ids_for_person_ids)
        predicted_class = knn_id_n.get(person_id, "Unknown")
        print(predicted_class)

        # Draw bounding box and label on the image
        cv.rectangle(small_frame, (left, top), (right, bottom), (0, 0, 255), 2)
        

cv.imshow('Frame', small_frame)
cv.waitKey(0)
cv.destroyAllWindows()



[1.0442721  1.0442721  1.0510033  ... 0.91530526 1.0474929  1.0527366 ]
1220 except6 condition
Haji Muhammad(Ex-Cheif of Section P&D)


In [11]:
p_id=['1220','1238','1264','1266']

In [12]:
face_path=["/home/devp/dataset/1220/11260.jpg", "/home/devp/dataset/1238/9076.jpg", "/home/devp/dataset/1264/486_.jpg", "/home/devp/dataset/1266/16332.jpg"]

In [13]:
face_embeddings=[]
for img_path in face_path:
    frame=cv.imread(img_path)
    small_frame = cv.resize(frame, (0, 0), fx=0.65, fy=0.65, interpolation=cv.INTER_AREA)
    origin_h, origin_w = small_frame.shape[:2]
    blob = cv.dnn.blobFromImage(small_frame)
    net.setInput(blob)
    detections = net.forward()
    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > thresholdfordetection:
            bounding_box = detections[0, 0, i, 3:7] * np.array([origin_w, origin_h, origin_w, origin_h])
            left, top, right, bottom = bounding_box.astype('int')
            face_frame = np.ascontiguousarray(frame[max(0, top-60):min(origin_h, bottom+60), max(0, left-60):min(origin_w, right+60)])
            
            pil_img = Image.fromarray(face_frame)
            transformed_img = make_transform(is_train=False)(pil_img).unsqueeze(0)  # Apply transformation
    
            # Extract embeddings from the model
            embedding = model(transformed_img).detach().cpu().numpy()[0]
            face_embeddings.append(embedding)
        

In [14]:
# face_embeddings[0]

In [15]:

frame = cv.imread("/home/devp/dataset/1266/7247_.jpg") 
   
    
small_frame = cv.resize(frame, (0, 0), fx=0.65, fy=0.65, interpolation=cv.INTER_AREA)
origin_h, origin_w = small_frame.shape[:2]
blob = cv.dnn.blobFromImage(small_frame)
net.setInput(blob)
detections = net.forward()

face_locations = []
face_encodings = []
face_names = []

for i in range(0, detections.shape[2]):
    confidence = detections[0, 0, i, 2]
    if confidence > thresholdfordetection:
        bounding_box = detections[0, 0, i, 3:7] * np.array([origin_w, origin_h, origin_w, origin_h])
        left, top, right, bottom = bounding_box.astype('int')
        face_frame = np.ascontiguousarray(frame[max(0, top-60):min(origin_h, bottom+60), max(0, left-60):min(origin_w, right+60)])
        
        pil_img = Image.fromarray(face_frame)
        transformed_img = make_transform(is_train=False)(pil_img).unsqueeze(0)  # Apply transformation


        # Extract embeddings from the model
        embedding = model(transformed_img).detach().cpu().numpy()[0]

        face_locations.append((left, top, right, bottom))
        face_encodings.append(embedding)

        # Use KNN to classify the face
        person_id, _ = knn_classifier(embedding, face_embeddings, p_id)
        predicted_class = knn_id_n.get(person_id, "Unknown")

        # Draw bounding box and label on the image
        cv.rectangle(small_frame, (left, top), (right, bottom), (0, 0, 255), 2)
        

cv.imshow('Frame', small_frame)
cv.waitKey(0)
cv.destroyAllWindows()



[0.84484595 1.1238261  1.0124367  0.9331702 ]
1220 except6 condition
